# Nettoyage et transformation des données

#### 1. Chargement des données en mémoire vive

A noter: l'encodage est automatiquement détecté par PySpark, pas besoin de réaliser de transformations pour les accents

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, StringType, DateType

bronze_path = "../data/water_quality_2026.json"
spark = SparkSession.builder.appName("RawJSON").getOrCreate()
df_raw = spark.read.json(bronze_path) # Détection automatique de l'encodage et du schéma
print("✅ Données brutes chargées depuis JSON")

✅ Données brutes chargées depuis JSON


#### 2. Définition des fonctions de nettoyage et de transformation


In [23]:
def clean_date(date_str):
    """Converts ISO string to DateType"""
    if not date_str:
        return None
    return date_str.split("T")[0]

def parse_numeric_result(text_val):
    """
    Parses strings like '<0,005' or '0,010' into floats.
    Handles decimal comma.
    """
    if text_val is None:
        return None
    
    text = str(text_val).replace(",", ".").strip()
    
    if text.startswith("<"):
        # Si "<0.005", on le traite comme 0.005 pour les analyses quantitatives,
        # mais selon la logique métier, on pourrait aussi choisir de le traiter comme une valeur manquante 
        try:
            return float(text[1:])
        except ValueError:
            return None
    else:
        try:
            return float(text)
        except ValueError:
            return None

def categorize_parameter(param_name):
    """Logique de catégorisation basée sur des mots-clés dans le nom du paramètre"""
    if not param_name:
        return "Autre"
    
    p = str(param_name).lower()
    
    if any(x in p for x in ["bactérie", "coli", "enterocoque", "coliforme", "spore"]):
        return "Microbiologie"
    elif any(x in p for x in ["nitrate", "nitrite", "pesticide", "atrazine", "glyphosate", "terbutylazine"]):
        return "Chimie_Pesticides"
    elif any(x in p for x in ["plomb", "cuivre", "aluminium", "chrome", "baryum", "fer", "magnésium"]):
        return "Chimie_Metaux"
    elif "ph" in p or "conductivité" in p or "température" in p:
        return "Physico-Chimie"
    elif "radio" in p or "tritium" in p or "dose" in p:
        return "Radioactivité"
    else:
        return "Autre"

#### 3. Appliquer une série de transformations pour rendre les données exploitables :

**A. Standardisation des Formats**
-   **Dates** : Conversion des timestamps ISO complexes en dates simples et extraction de l'année pour le partitionnement futur.
-   **Valeurs Numériques** : Le fichier contient des formats mixtes (ex: `<0,005` ou `0,010`). Nous utilisons une logique de parsing pour extraire la valeur numérique réelle, en traitant les seuils de détection (`<`) comme des valeurs limites.
-   **Noms de colonnes** : Uniformisation pour faciliter les requêtes futures.

In [24]:
# Nettoyage et conversion de la date
df_clean = df_raw.withColumn(
    "date_prelevement_clean", 
    F.to_date(F.col("date_prelevement"))
).withColumn(
    "annee", 
    F.year(F.col("date_prelevement_clean"))
)

# Output: Affichage d'un échantillon des dates nettoyées et des années extraites
# print("Echantillon: Date brute -> Date nettoyée -> Année")
# df_clean.select(
#     "date_prelevement", 
#     "date_prelevement_clean", 
#     "annee"
# ).limit(5).show(truncate=False)

# Vérification de la distribution des années
# print("\nRépartition des années:")
# df_clean.groupBy("annee").count().orderBy("annee").show()

In [25]:
# Traitement des résultats alphanumériques
df_clean = df_clean.withColumn(
    "resultat_valeur_float",
    F.udf(parse_numeric_result, FloatType())(F.col("resultat_alphanumerique"))
)

# Output: Affichage d'un échantillon de la comparaison entre le texte alphanumérique et la valeur flottante parsée
print("Echantillon: Texte alphanumérique -> Valeur flottante parsée")
df_clean.select(
    "libelle_parametre",
    "resultat_alphanumerique", 
    "resultat_valeur_float",
    "libelle_unite"
).limit(10).show(truncate=False)

# Vérification des cas où le parsing a échoué (valeur alphanumérique non nulle mais valeur float nulle)
failed_parse = df_clean.filter(
    (F.col("resultat_alphanumerique").isNotNull()) & 
    (F.col("resultat_valeur_float").isNull())
).count()
print(f"\n⚠️  Lignes avec échec de parsing: {failed_parse}")

Echantillon: Texte alphanumérique -> Valeur flottante parsée
+----------------------------------+-----------------------+---------------------+-------------+
|libelle_parametre                 |resultat_alphanumerique|resultat_valeur_float|libelle_unite|
+----------------------------------+-----------------------+---------------------+-------------+
|Conductivité à 25°C               |735                    |735.0                |µS/cm        |
|Température de l'eau              |11                     |11.0                 |°C           |
|pH                                |7,4                    |7.4                  |unité pH     |
|Température de mesure du pH       |10,7                   |10.7                 |°C           |
|Entérocoques /100ml-MS            |0                      |0.0                  |n/(100mL)    |
|Escherichia coli /100ml - MF      |0                      |0.0                  |n/(100mL)    |
|Bact. aér. revivifiables à 36°-44h|>300                   |NULL  


⚠️  Lignes avec échec de parsing: 16696


Standardisation des Noms de Colonnes

Uniformisation des noms de colonnes pour faciliter les requêtes futures et rendre les données plus exploitables.

In [26]:
df_clean = df_clean.withColumnRenamed("libelle_parametre", "parametre_nom") \
                   .withColumnRenamed("nom_commune", "commune") \
                   .withColumnRenamed("nom_departement", "departement") \
                   .withColumnRenamed("code_departement", "code_dept")

print("✅ Noms de colonnes standardisés")

✅ Noms de colonnes standardisés


**B. Enrichissement Sémantique**
Au lieu de travailler avec des centaines de noms de paramètres techniques, nous regroupons automatiquement les mesures en **5 familles de risques** basées sur leur nom :
1.  **Microbiologie** (Bactéries, E. coli)
2.  **Chimie - Pesticides** (Nitrates, Glyphosate, Atrazine...)
3.  **Chimie - Métaux** (Plomb, Cuivre, Aluminium...)
4.  **Physico-Chimie** (pH, Conductivité, Température)
5.  **Radioactivité** (Dose indicative, Tritium)

Cette étape transforme des données techniques en informations décisionnelles.

In [27]:
# Catégorisation des paramètres
df_clean = df_clean.withColumn(
    "categorie",
    F.udf(categorize_parameter)(F.col("parametre_nom"))
)

# Output: Affichage d'un échantillon des paramètres avec leurs catégories assignées
print("Echantillon: Parameter Name -> Assigned Category")
df_clean.select(
    "parametre_nom", 
    "categorie"
).distinct().limit(15).show(truncate=False)

# Décompte du nombre de mesures par catégorie pour vérifier la répartition
print("\nCount by Category:")
df_clean.groupBy("categorie").count().orderBy(F.desc("count")).show()



Echantillon: Parameter Name -> Assigned Category


+-------------------------------------------------+-----------------+
|parametre_nom                                    |categorie        |
+-------------------------------------------------+-----------------+
|Buturon                                          |Autre            |
|Imazaquine                                       |Autre            |
|Picoxystrobine                                   |Autre            |
|Chlorpyriphos méthyl                             |Physico-Chimie   |
|Terbutryne                                       |Autre            |
|Pyributicarb                                     |Autre            |
|Chlorites en cas de traitement pouvant en générer|Autre            |
|Fonofos                                          |Autre            |
|Ethoxysulfuron                                   |Autre            |
|Atrazine-déisopropyl                             |Chimie_Pesticides|
|Metalaxyl CGA 108906                             |Autre            |
|Crésol para        

+-----------------+------+
|        categorie| count|
+-----------------+------+
|            Autre|204960|
|   Physico-Chimie| 35609|
|Chimie_Pesticides| 18249|
|    Microbiologie| 13908|
|    Chimie_Metaux|  6692|
|    Radioactivité|   582|
+-----------------+------+



**C. Nettoyage et Déduplication**
Les mêmes prélèvements peuvent parfois apparaître plusieurs fois. Nous filtrons les données invalides et conservons une seule entrée par combinaison unique : `ID Prélèvement + Paramètre + Date`.

In [28]:
# Filtrage final pour ne garder que les lignes avec les champs essentiels non nuls
df_final = df_clean.filter(
    (F.col("code_prelevement").isNotNull()) &
    (F.col("parametre_nom").isNotNull()) &
    (F.col("commune").isNotNull())
)

# Déduplication basée sur les champs clés (code prélevement, code paramètre, date de prélèvement)
rows_before = df_final.count()
df_final = df_final.dropDuplicates(
    subset=["code_prelevement", "code_parametre", "date_prelevement_clean"]
)
rows_after = df_final.count()

# Output: Affichage de statistiques de nettoyage et déduplication
print(f"Lignes avant déduplication: {rows_before}")
print(f"Lignes après déduplication:  {rows_after}")
print(f"Lignes supprimées:        {rows_before - rows_after}")

# Affichage du schéma final et d'un échantillon de données nettoyées
print("\nÉchantillon de données nettoyées:")
df_final.select(
    "code_prelevement", "commune", "departement", "parametre_nom", 
    "resultat_valeur_float", "categorie", "annee"
).limit(5).show(truncate=False)

Lignes avant déduplication: 280000
Lignes après déduplication:  277813
Lignes supprimées:        2187

Échantillon de données nettoyées:


+----------------+-----------------+-----------+----------------------------------+---------------------+-----------------+-----+
|code_prelevement|commune          |departement|parametre_nom                     |resultat_valeur_float|categorie        |annee|
+----------------+-----------------+-----------+----------------------------------+---------------------+-----------------+-----+
|00100149297     |AMBERIEU-EN-BUGEY|Ain        |Conductivité à 25°C               |415.0                |Physico-Chimie   |2026 |
|00100149297     |AMBERIEU-EN-BUGEY|Ain        |Nitrates (en NO3)                 |5.4                  |Chimie_Pesticides|2026 |
|00100149297     |AMBERIEU-EN-BUGEY|Ain        |Entérocoques /100ml-MS            |1.0                  |Autre            |2026 |
|00100149298     |ROSSILLON        |Ain        |Température de l'eau              |12.0                 |Physico-Chimie   |2026 |
|00100149298     |ROSSILLON        |Ain        |Bact. aér. revivifiables à 22°-68h|21.0   

#### 4. Export couche Silver intermédiaire (parquet)

In [29]:
# Ecriture dans un répertoire local
# Création du répertoire de sortie s'il n'existe pas
import os
os.makedirs("..data/silver", exist_ok=True)

output_path = "../data/silver/water_quality_clean"
df_final.write.mode("overwrite").parquet(output_path)

print(f"\n✅ Données argent sauvegardées dans: {output_path}")

# Quick verification
df_verify = spark.read.parquet(output_path)
df_verify.groupBy("categorie").count().show()


26/05/05 12:22:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/05 12:22:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/05 12:22:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/05 12:22:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/05 12:22:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/05 12:22:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/05 12:22:05 WARN MemoryManager: Total allocation exceeds 95.


✅ Données argent sauvegardées dans: ../data/silver/water_quality_clean
+-----------------+------+
|        categorie| count|
+-----------------+------+
|    Microbiologie| 13892|
|Chimie_Pesticides| 18239|
|   Physico-Chimie| 33506|
|    Chimie_Metaux|  6669|
|    Radioactivité|   582|
|            Autre|204925|
+-----------------+------+

